# Guided Diffusion Model — Prior-Driven Drift Fitting

Fits the guided diffusion memory model (Hypothesis 2: Prior-Based Account).
Memory traces drift toward prior modes via the score function ∇ log π(x),
implementing Langevin dynamics guided by a learned texture prior.

**Key difference from the three-regime model:** No noise regimes. A single
noise parameter σ₀ governs all memory noise, plus `drift_step_size` controls
the strength of prior-driven drift.

| Stage | Parameter | Method | ISIs |
|-------|-----------|--------|------|
| A | `sigma0` (encoding noise) | ISI-0 toy experiments | [0] |
| B | `drift_step_size` | Grid search on multi-ISI sequences | [1, 2, 4, 8, 16, 32, 64] |

In [2]:
import sys, os, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict
from functools import partial
from scipy.spatial.distance import pdist

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.params import model_params as model_params_tm

from texture_prior.utils import path

from utls.plotting import ensure_dir
from utls.loading import (
    load_results_with_exclusion_2,
    move_sequences_to_used,
    load_results_with_exclusion_no_dropping,
)
from utls.runners_v2 import (
    run_experiment_scores_prior,
    make_noise_schedule,
)
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.io_utils import (
    make_model_save_dir,
    save_all_figures,
    save_single_figure,
    save_runs_summary,
)
from encoders import *

from utls.toy_experiments import (
    make_toy_experiment_list,
    make_compact_multi_isi_sequences,
    infer_trial_isis,
    make_high_diversity_sequences,
)
from utls.sigma_fitting import (
    log_mid,
    fit_sigma_1d,
    plot_sigma_fit,
    _compute_auroc_upper_envelope,
    auc_to_dprime as auc2dp,
)

from ScoreFunction import ScoreFunction

## 1. Load config & data

In [3]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path


# --- Use a texture-based config ---
CONFIG_PATH = (
    "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)

model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)

{'results_root': '/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory', 'tag': 'slurm', 'experiment': {'is_multi': True, 'n_seqs': 36, 'n_samples': 50, 'which_task': 0}, 'metric': 'cosine', 'noise_model': {'name': 'three-regime', 'sigma0_min': 3.0, 'sigma0_max': 0.5, 'sigma1_min': 0.1, 'sigma1_max': 0.6, 'sigma2_min': 0.0005, 'sigma2_max': 0.1, 't_step': 5}, 'run_id': 'run_000005', 'representation': {'type': 'resnet50', 'layer': 'layer4', 'time_avg': False}}


In [4]:
# ---- experiment ----
exp_cfg = model_cfg["experiment"]
which_task = 2
is_multi = exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)

isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model: constant (single regime) for this model ----
noise_mode = "constant"

# ---- representation: texture PCA 256D ----
repr_cfg = "texture_pca"
time_avg = False
encoder_type = "texture"
layer = None
pc_dims = 256

# ---- load human data ----
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(2, 0, True, old=False)
)

human_curve = compute_human_curve(human_runs, is_multi, which_isi)
print("ISIs:", isis)
print("Human d' curve:", human_curve)
print(f"Total real sequences: {len(exp_list)}")

/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/runners_utils.py:210: RuntimeWarning: Mean of empty slice
  dprimes.append(np.nanmean(aucs))


ISIs: [0, 1, 2, 4, 8, 16, 32, 64]
Human d' curve: [2.16716439 1.83069423 1.3296253  1.20497243 1.07817307 0.99154791
 0.97911993 0.85841501]
Total real sequences: 120


## 2. Build encoder & encode stimuli

In [5]:
#all_files = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")

pc_dims = 256
device = 'cuda'


pc_texture_model = AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.01,
    duration=2.0,
    device=device
)


X = encode_stimuli(pc_texture_model, all_files)
X_np = X.detach().cpu().numpy()
print("Encoded shape:", X_np.shape, "  any NaN?", torch.isnan(X).any().item())

Encoded shape: (80, 256)   any NaN? False


## 3. Load score function (prior)

In [6]:
SCORE_CONFIG = "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/bryan.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

score_fn = ScoreFunction(
    mode="textures",
    restart=True,
    config=SCORE_CONFIG,
    device=DEVICE,
)

# Quick sanity check: forward pass on a random input
test_input = torch.randn(1, 256, device=DEVICE)
test_output = score_fn.forward(test_input)
print(f"Score output shape: {test_output.shape}")
print(f"Score output norm: {test_output.view(-1).norm():.4f} (should be ~1.0)")

Score output shape: torch.Size([1, 1, 1, 256])
Score output norm: 1.0000 (should be ~1.0)


## 4. Parameter bounds & stimulus pool

In [7]:
param_bounds = {
    "sigma0": (0.0001, 5),
    "sigma": (0.0001, 10),
    "drift_step_size": (0.01, 1.0),
}

print("Parameter bounds:")
for k, v in param_bounds.items():
    print(f"  {k}: ({v[0]:.6f}, {v[1]:.6f})")

# Stimulus pool for sequence generation
stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f"\nStimulus pool size: {len(stimulus_pool)}")
assert len(stimulus_pool) >= 65, (
    f"Stimulus pool ({len(stimulus_pool)}) too small for ISI-64 blocks (need >= 65)"
)

Parameter bounds:
  sigma0: (0.000100, 5.000000)
  sigma: (0.000100, 10.000000)
  drift_step_size: (0.010000, 1.000000)

Stimulus pool size: 80


## 5. Human d' targets

In [8]:
isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}

sigma0_human = {0: float(human_curve[isi_to_hc_idx[0]])}
sigma_human = {
    isi: float(human_curve[isi_to_hc_idx[isi]])
    for isi in [1, 2, 4, 8, 16, 32, 64]
}

print("Stage A targets (sigma0 — encoding noise):")
for isi, dp in sigma0_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

print("\nStage B targets (sigma — drift noise):")
for isi, dp in sigma_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

# Initial values
sigma_init = log_mid(*param_bounds["sigma"])
drift_init = log_mid(*param_bounds["drift_step_size"])
print(f"\nInitial sigma = {sigma_init:.6f}")
print(f"Initial drift_step_size = {drift_init:.6f}")

Stage A targets (sigma0 — encoding noise):
  ISI 0: human d' = 2.1672

Stage B targets (sigma — drift noise):
  ISI 1: human d' = 1.8307
  ISI 2: human d' = 1.3296
  ISI 4: human d' = 1.2050
  ISI 8: human d' = 1.0782
  ISI 16: human d' = 0.9915
  ISI 32: human d' = 0.9791
  ISI 64: human d' = 0.8584

Initial sigma = 0.031623
Initial drift_step_size = 0.100000


---
## Stage A: Fit sigma0 (ISI = 0, toy experiments)

ISI=0 means immediate repeats — only one step of noise, so only sigma0
matters. Standard toy experiments (pairs of identical stimuli) are
efficient here. Drift has no effect at ISI=0 (no intervening items).

In [ ]:
isi0_exps = {
    0: make_toy_experiment_list(
        stimulus_pool, isi=0, n_experiments=50, k_stimuli=5, seed=0
    )
}
print(f"ISI-0 toy experiments: {len(isi0_exps[0])} exps, "
      f"avg len {np.mean([len(e) for e in isi0_exps[0]]):.0f} trials")

N_REFINE_ITERS = 2
N_MC = 8

# For stage A, drift is negligible, but we pass it for API consistency
stage_a = fit_sigma_1d(
    run_experiment_fn=run_experiment_scores_prior,
    sigma_name="sigma0",
    sigma_bounds=param_bounds["sigma0"],
    fixed_sigmas={"sigma0": sigma_init},  # sigma0 = drift noise, overridden by grid
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiments_by_isi=isi0_exps,
    human_dprimes_by_isi=sigma0_human,
    t_step=None,
    n_grid=15,
    n_mc=N_MC,
    n_refine_iters=N_REFINE_ITERS,
    seed=0,
    score_model=score_fn
)

sigma0_fitted = stage_a["best_sigma"]
print(f"\n>>> sigma0 = {sigma0_fitted:.6f}  (MSE = {stage_a['best_mse']:.6f})")
plot_sigma_fit(stage_a, human_dprimes_by_isi=sigma0_human,
               title="Stage A: sigma0 (ISI = 0)");

ISI-0 toy experiments: 50 exps, avg len 10 trials

--- sigma0 iteration 1/2 ---
  Bounds: (0.000100, 5.000000), 15 candidates


Fitting sigma0 (iter 1):   0%|          | 0/15 [00:00<?, ?it/s]

evaluate_sigma_on_multi_isi_sequences_sample
evaluate_sigma_on_multi_isi_sequences_sample
evaluate_sigma_on_multi_isi_sequences_sample
evaluate_sigma_on_multi_isi_sequences_sample


---
## Stage B: Fit drift_step_size with fixed sigma0

Using compact multi-ISI sequences covering ISIs [1, 2, 4, 8, 16, 32, 64].
We fix sigma0 to the Stage A value and grid-search over drift_step_size
to match the human d' curve across ISIs.

In [1]:
SIGMA_ISIS = [1,2]
SIGMA_LENGTH = 30
SIGMA_N_SEQS = 5
SIGMA_MIN_PAIRS = 5

sigma_exps, sigma_isi_keys = make_high_diversity_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=SIGMA_ISIS,
    n_sequences=SIGMA_N_SEQS,
    length=SIGMA_LENGTH,
    min_pairs_per_isi=SIGMA_MIN_PAIRS,
    seed=1000,
)

print(f"Generated {len(sigma_exps)} compact sequences")
print(f"Trials per sequence: {len(sigma_exps[0])}")

# Validate ISI distribution
sigma_isi_counts = defaultdict(list)
for seq in sigma_exps:
    counts = Counter(infer_trial_isis(seq))
    for isi_val in SIGMA_ISIS:
        sigma_isi_counts[isi_val].append(counts.get(isi_val, 0))

print("\nPairs per ISI per sequence (mean +/- std):")
for isi_val in SIGMA_ISIS:
    vals = sigma_isi_counts[isi_val]
    print(f"  ISI {isi_val:>2}: {np.mean(vals):.1f} +/- {np.std(vals):.1f}  "
          f"(min={min(vals)}, max={max(vals)})")

NameError: name 'make_high_diversity_sequences' is not defined

In [ ]:
# Alternating coordinate descent over (sigma, drift_step_size),
# holding sigma0 fixed from Stage A.

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

SIGMA_BOUNDS = param_bounds["sigma"]
DRIFT_BOUNDS = param_bounds["drift_step_size"]
N_SIGMA_GRID = 10
N_DRIFT_GRID = 10
N_MC = 10          # MC reps per grid point
N_SEQS_PER_REP = 5
N_COORD_ITERS = 3  # alternating iterations

sigma_grid = np.geomspace(SIGMA_BOUNDS[0], SIGMA_BOUNDS[1], N_SIGMA_GRID)
drift_grid = np.geomspace(DRIFT_BOUNDS[0], DRIFT_BOUNDS[1], N_DRIFT_GRID)
seq_trial_isis = [infer_trial_isis(seq) for seq in sigma_exps]


def evaluate_params(sigma_val, drift_val):
    """Evaluate a (sigma, drift_step_size) pair.  Returns (mean_mse, dprime_by_isi)."""
    run_fn = partial(
        run_experiment_scores_prior,
        score_model=score_fn,
        drift_step_size=drift_val,
        sigma=sigma_val,
    )
    mse_reps = []
    dprime_reps = {isi: [] for isi in SIGMA_ISIS}

    for rep in range(N_MC):
        rng = np.random.default_rng(200_000 + rep)
        seq_idx = rng.choice(len(sigma_exps), size=N_SEQS_PER_REP, replace=False)

        all_hits, all_isis_for_hits, all_fas = [], [], []
        for si in seq_idx:
            run_out = run_fn(
                sigma0=sigma0_fitted,
                noise_mode=noise_mode,
                metric=metric,
                X0=X,
                name_to_idx=name_to_idx,
                experiment_list=[sigma_exps[si]],
                debug=False,
                seed=200_000 + rep * 1000 + int(si),
            )
            h = np.asarray(run_out["hits"])
            f = np.asarray(run_out["fas"])
            t_isis = seq_trial_isis[si]
            if len(h) != len(t_isis):
                continue
            all_hits.append(h)
            all_isis_for_hits.extend(t_isis)
            all_fas.append(f)

        if not all_hits:
            continue
        hits_arr = np.concatenate(all_hits)
        isis_arr = np.array(all_isis_for_hits)
        fas_arr = np.concatenate(all_fas) if all_fas else np.array([])
        if len(fas_arr) == 0:
            continue

        rep_mse = []
        for isi_val in SIGMA_ISIS:
            mask = isis_arr == isi_val
            hits_isi = hits_arr[mask]
            if len(hits_isi) == 0:
                continue
            human_dp = sigma_human.get(isi_val)
            if human_dp is None:
                continue
            auroc = _compute_auroc_upper_envelope(hits_isi, fas_arr)
            dp = auc2dp(auroc, eps=1e-4)
            dprime_reps[isi_val].append(dp)
            rep_mse.append((dp - human_dp) ** 2)

        if rep_mse:
            mse_reps.append(np.mean(rep_mse))

    mean_mse = np.mean(mse_reps) if mse_reps else np.inf
    mean_dprimes = {isi: np.mean(v) if v else np.nan for isi, v in dprime_reps.items()}
    return mean_mse, mean_dprimes


# --- Alternating coordinate descent ---
current_sigma = sigma_init
current_drift = drift_init
best_sigma, best_drift, best_mse = current_sigma, current_drift, np.inf
all_results = []

with torch.no_grad():
    for coord_iter in range(N_COORD_ITERS):
        print(f"\n{'='*50}")
        print(f"Coordinate descent iteration {coord_iter + 1}/{N_COORD_ITERS}")
        print(f"{'='*50}")

        # Step 1: Fix drift, sweep sigma
        print(f"\n  Sweeping sigma (drift_step_size fixed at {current_drift:.6f})")
        iter_best_sigma, iter_best_mse_sigma = current_sigma, np.inf
        for s in tqdm(sigma_grid, desc=f"  sigma (iter {coord_iter+1})"):
            mse, dprimes = evaluate_params(s, current_drift)
            all_results.append({
                "iter": coord_iter, "phase": "sigma",
                "sigma": s, "drift_step_size": current_drift,
                "mse_mean": mse, "dprime_by_isi": dprimes,
            })
            if mse < iter_best_mse_sigma:
                iter_best_mse_sigma = mse
                iter_best_sigma = s
            torch.cuda.empty_cache()

        current_sigma = iter_best_sigma
        print(f"  >> best sigma = {current_sigma:.6f}  (MSE = {iter_best_mse_sigma:.6f})")

        # Step 2: Fix sigma, sweep drift
        print(f"\n  Sweeping drift_step_size (sigma fixed at {current_sigma:.6f})")
        iter_best_drift, iter_best_mse_drift = current_drift, np.inf
        for ds in tqdm(drift_grid, desc=f"  drift (iter {coord_iter+1})"):
            mse, dprimes = evaluate_params(current_sigma, ds)
            all_results.append({
                "iter": coord_iter, "phase": "drift",
                "sigma": current_sigma, "drift_step_size": ds,
                "mse_mean": mse, "dprime_by_isi": dprimes,
            })
            if mse < iter_best_mse_drift:
                iter_best_mse_drift = mse
                iter_best_drift = ds
            torch.cuda.empty_cache()

        current_drift = iter_best_drift
        print(f"  >> best drift = {current_drift:.6f}  (MSE = {iter_best_mse_drift:.6f})")

        # Track overall best
        if iter_best_mse_drift < best_mse:
            best_mse = iter_best_mse_drift
            best_sigma = current_sigma
            best_drift = current_drift

print(f"\n>>> sigma = {best_sigma:.6f},  drift_step_size = {best_drift:.6f}  (MSE = {best_mse:.6f})")

# --- Plot final iteration results ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Sigma sweep (last iteration)
last_sigma_results = [r for r in all_results if r["iter"] == N_COORD_ITERS - 1 and r["phase"] == "sigma"]
if last_sigma_results:
    s_vals = [r["sigma"] for r in last_sigma_results]
    s_mse = [r["mse_mean"] for r in last_sigma_results]
    axes[0].plot(s_vals, s_mse, 'o-')
    axes[0].axvline(best_sigma, color='red', ls='--', alpha=0.7, label=f'best={best_sigma:.4f}')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('sigma')
    axes[0].set_ylabel('MSE vs human d\'')
    axes[0].set_title(f'sigma sweep (iter {N_COORD_ITERS})')
    axes[0].legend()
    axes[0].grid(alpha=0.25)

# Drift sweep (last iteration)
last_drift_results = [r for r in all_results if r["iter"] == N_COORD_ITERS - 1 and r["phase"] == "drift"]
if last_drift_results:
    ds_vals = [r["drift_step_size"] for r in last_drift_results]
    ds_mse = [r["mse_mean"] for r in last_drift_results]
    axes[1].plot(ds_vals, ds_mse, 'o-')
    axes[1].axvline(best_drift, color='red', ls='--', alpha=0.7, label=f'best={best_drift:.4f}')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('drift_step_size')
    axes[1].set_ylabel('MSE vs human d\'')
    axes[1].set_title(f'drift sweep (iter {N_COORD_ITERS})')
    axes[1].legend()
    axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

---
## 6. Fitted parameter summary

In [ ]:
print("=" * 50)
print("GUIDED DIFFUSION MODEL FIT")
print("=" * 50)
print(f"  sigma0 (encoding)    = {sigma0_fitted:.6f}  (Stage A MSE: {stage_a['best_mse']:.6f})")
print(f"  sigma  (Langevin)    = {best_sigma:.6f}  (Stage B MSE: {best_mse:.6f})")
print(f"  drift_step_size      = {best_drift:.6f}")

---
## 7. Final evaluation on ALL real sequences

Evaluate the fitted parameters on every real participant sequence.

In [ ]:
%%time

# Run the fitted model on the real experiment sequences
with torch.no_grad():
    final_run = run_experiment_scores_prior(
        sigma0=sigma0_fitted,
        sigma=best_sigma,
        noise_mode=noise_mode,
        metric=metric,
        X0=X,
        name_to_idx=name_to_idx,
        experiment_list=exp_list,
        score_model=score_fn,
        drift_step_size=best_drift,
        seed=42,
    )

model_dp = compute_model_dprime_for_run(final_run, isis)

print("Model d' curve:", model_dp)
print("Human d' curve:", human_curve)

## 8. Summary plot: model vs human d' across all ISIs

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

isi_labels = [str(i) for i in isis]
x_pos = np.arange(len(isis))

ax.plot(x_pos, human_curve, 'ko-', label='Human', linewidth=2, markersize=8)
ax.plot(x_pos, model_dp, 's--', color='tab:blue',
        label=f'Guided Diffusion (σ₀={sigma0_fitted:.3f}, σ={best_sigma:.4f}, drift={best_drift:.4f})',
        linewidth=2, markersize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels(isi_labels)
ax.set_xlabel('ISI (intervening items)')
ax.set_ylabel("d'")
ax.set_title('Guided Diffusion Model vs Human Performance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()